In [11]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding, DataCollatorForSeq2Seq
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

from datasets import load_dataset, load_from_disk

import torch
from torch.utils.data import DataLoader

import evaluate
from sklearn.metrics import accuracy_score, f1_score

from tqdm import tqdm

In [2]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [3]:
torch.cuda.is_available()


True

# Task-A: Sentiment Analysis

## A.1 Pretrained model
In this subsection we will load a pretrained model and evaluate its performance on a movie, sentiment analysis task.

In Hugging Face’s Transformers library, the `Auto Classes` are generic wrappers that automatically pick the right model/tokenizer/config class for you, based on the pretrained model you’re loading.

### AutoTokenizer
- `AutoTokenizer` is a Hugging Face class that automatically picks the right tokenizer for the model you specify.
- A tokenizer is responsible for splitting text into tokens (subwords or word pieces) that the model can understand.
- `AutoTokenizer.from_pretrained(model_name)` downloads and loads the pretrained tokenizer for the model of your choice.

### AutoModelForSequenceClassification
- `AutoModelForSequenceClassification` is a Hugging Face class that loads a model configured for sequence classification (e.g., sentiment analysis, text classification).
- `AutoModelForSequenceClassification.from_pretrained(model_name)`This method loads a model that has already been trained (pretrained) and published.

This is the setup we’re using for the following exercise. In practice, Hugging Face provides many additional classes for a wide variety of setups. You are encouraged to explore the [Auto Classes documentation](https://huggingface.co/docs/transformers/en/model_doc/auto) to get a broader understanding of what’s available.

**`TODO:`** Load the tokenizer and the pretrained model for sequence classification for `distilbert/distilbert-base-uncased`.

In [4]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Datasets
- The datasets library is another core library from Hugging Face, separate from transformers.
- In short, this library is designed to make it easy to access, share, preprocess, and work with large datasets (especially for NLP, but also vision, audio, and multimodal tasks).
- The `load_dataset("dataset_name")` function pulls and prepares a dataset from the internet.
- The `load_from_disk("dataset_name")` function loads a local dataset in the same way.
- The `.map()` method applies a function to every element (or batch of elements) in a Dataset. Have a look at an example here: [link](https://huggingface.co/docs/datasets/en/process#map).
- When loading the dataset using the above function, the `split` argument can be used to get a specific split.
- Note: These are different from the PyTorch datasets. They're similar and it's easy to transition from one to the other but they're not identical.

For further documentation regarding HF dataset, you are encouraged to explore the following [documentation](https://huggingface.co/docs/datasets/en/index).

**`TODO:`** Load the train and test split of the `imdb` dataset. How many samples are in each split?

In [5]:

dataset = load_dataset('imdb')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### DataLoader
- `DataLoader` is a PyTorch utility that wraps a dataset and handles batching, shuffling, and parallel loading.
- It takes a dataset (can be from both HF and PyTorch) and returns an iterator you can loop over in training or evaluation.

Have a look at its [documentation](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader). Specifically, see what arguments are need to load a dataset, to set your own batch size and decide whether the data will be shuffled or not.

**`TODO:`** Load your test data into a `DataLoader` of batch size 16. Do not shuffle the data.

In [6]:
test_loader = DataLoader(dataset['test'],batch_size=16,shuffle=False)

**`TODO:`** As we have previously mentioned, `DataLoader` returns an iterator. Using a `for` loop, investigate what the iterator returns for its first iteration.

In [7]:
for batch in test_loader:
    print(batch)
    break  # 只看第一个 batch


{'text': ['I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as

In [8]:

dataset['test']

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

### Evaluation function

Training a model is only half the story—what really matters is how well it performs on data it hasn’t seen before. That’s where evaluation comes in.

We’ll take a pretrained Hugging Face model and a test dataset (examples the model hasn’t trained on) and run the model’s predictions against the true labels. The steps we’ll implement are:

1. Preprocessing the data: Tokenize the input text so it can be understood by the model.
2. Forward pass: Feed the tokens through the model to get predicted outputs (logits).
3. Predictions: Convert the logits into actual predicted labels (positive/negative sentiment).
4. Metrics: Compare predictions to the true labels using metrics like:
    - Accuracy: How often the model is correct overall.
    - F1 Score: A balance of precision and recall, useful when the dataset is imbalanced.
    - Positive vs. Negative Accuracy: How well the model handles each class individually.

**`TODO:`** Complete all the missing code from the following function. Why do we sometimes look at Positive vs. Negative accuracy?

In [9]:
def evaluate_model(model, tokenizer, test_loader):
    """
    Evaluate a Hugging Face sequence classification model on a test dataset.

    Parameters:
        model (transformers.AutoModelForSequenceClassification): The pretrained sequence classification model to evaluate.
        tokenizer (transformers.AutoTokenizer): The tokenizer used to preprocess the input text for the model.
        test_loader (torch.utils.data.DataLoader): A DataLoader providing the test dataset.

    Returns:
        all_preds (torch.Tensor): Model predictions on the test set.
        all_labels (torch.Tensor): Ground truth labels from the test set.
        acc (float): Overall accuracy on the test set.
        f1 (float): F1 score on the test set.
    """

    all_labels = None
    all_preds = None

    for batch in tqdm(test_loader):
        text = batch['text']
        labels = batch['label']

        # TODO: Tokenize the text using the provided tokenizer, use both truncation and padding.
        inputs = tokenizer(text,truncation=True,padding=True,return_tensors="pt")


        if torch.cuda.is_available():
            inputs = inputs.to('cuda')
            model = model.to('cuda')

        # TODO: Perform a forward pass through the output of the model
        with torch.no_grad():
            outputs = model(**inputs)

        # TODO: Get the logits from the model's output and compute the predictions by taking the argmax
        logits = outputs.logits
        preds = logits.argmax(dim = -1)

        if all_labels is None:
            all_labels = labels.cpu()
            all_preds = preds.cpu()
        else:
            all_labels = torch.cat((all_labels, labels.cpu()))
            all_preds = torch.cat((all_preds, preds.cpu()))

    # TODO: compute f1 score between model predictions and ground-truth labels (you can use sklearn.metrics)
    f1 = f1_score(all_preds,all_labels)

    # TODO: compute accuracy score between model predictions and ground-truth labels (you can use sklearn.metrics)
    acc = accuracy_score(all_preds,all_labels)

    # TODO: compute the accuracy on Positive(label==1) samples
    pos_acc = accuracy_score(all_preds[all_labels==1],all_labels[all_labels==1])

    # TODO: compute the accuracy on Negative(label==0) samples
    neg_acc = accuracy_score(all_preds[all_labels==0],all_labels[all_labels==0])

    print('Accuracy: ', acc*100, '%')
    print(' -- Positive Accuracy: ', pos_acc*100, '%')
    print(' -- Negative Accuracy: ', neg_acc*100, '%')
    print('F1 score: ', f1)

    return all_preds, all_labels, acc, f1

## A.2 Finetuned model
In this section, we will further train the model for the task that we are interested in and see if we can increase its performance.

**`TODO:`** Use `evaluate_model` to measure the performance of the pretrained model.

In [12]:


evaluate_model(model,tokenizer,test_loader)

100%|██████████| 1563/1563 [06:18<00:00,  4.13it/s]

Accuracy:  50.0 %
 -- Positive Accuracy:  0.008 %
 -- Negative Accuracy:  99.992 %
F1 score:  0.00015997440409534473


(tensor([0, 0, 0,  ..., 0, 0, 0]),
 tensor([0, 0, 0,  ..., 1, 1, 1]),
 0.5,
 0.00015997440409534473)

**`TODO:`** Define a function that receives some samples and then uses the tokenizer we have defined to tokenize the samples. Use the `Dataset.map`method that we have previously discussed to apply your function to the train data. Keep this in mind when defining the tokenizing function.

In [13]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, return_tensors="pt")

### Data Collator
- A data collator is a small function/class that tells the DataLoader how to merge a list of individual samples into a single batch.
- In NLP, a main challenge is padding. Since sentences have different lengths, you need to pad them so they fit into a uniform tensor batch.
- The `DataCollatorWithPadding` will  automatically pad sequences in a batch to the length of the longest sequence in that batch (dynamic padding) based on the tokenizer you're using.

For more info on Data Collators please refer to the following [documentation](https://huggingface.co/docs/transformers/en/main_classes/data_collator#transformers.DataCollatorWithPadding).

**`TODO:`** Define a data collator that automatically pads sequences in a batch based on the defined `tokenizer`.

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Training Arguments
- A configuration object that stores all the knobs and settings related to the training procedure (like learning rate, batch size, number of epochs, output directory, etc.).
- It tells the training loop how to train (e.g., optimization settings, saving/checkpoint rules, logging options).
- You don’t train with it directly, you just define the "rules of training."

### Trainer
- This is the high-level training loop: it actually runs the training, evaluation, and prediction based on your model, data, and `TrainingArguments`.
- It takes care of all the heavy lifting (forward pass, loss calculation, backprop, optimizer steps, checkpoint saving, etc.).
- You just call methods like `.train()`, `.evaluate()`, or `.predict()` without writing a manual loop.

In [15]:
output_dir_name = "imdb-finetuned-distilbert"

training_args = TrainingArguments(
    output_dir=output_dir_name,          # Where to save model checkpoints and logs
    learning_rate=2e-5,                  # Step size for the optimizer (how fast the model learns)
    per_device_train_batch_size=16,      # Training batch size per GPU/CPU device
    per_device_eval_batch_size=16,       # Evaluation batch size per GPU/CPU device
    num_train_epochs=1,                  # Number of times to iterate over the full training dataset
    weight_decay=0.01,                   # Strength of L2 regularization (helps prevent overfitting)
    save_strategy="epoch",               # When to save checkpoints ("epoch" = at the end of each epoch)
    push_to_hub=False,                   # Whether to push the model to the Hugging Face Hub
    report_to="none"                     # Where to report logs (e.g., "wandb", "tensorboard", "none")
)

**`TODO:`** Based on everything what we have defined so far in this exercise, complete the following code to initialize the trainer.

In [18]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=dataset['train'],
                  eval_dataset=dataset['test'],
                  tokenizer=tokenizer,
                  data_collator=data_collator)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

/tmp/ipython-input-630003058.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


**`TODO:`** Use the `.train` method of the trainer to train the model

In [19]:
trainer.train()

Step,Training Loss
500,0.313000
1000,0.245100
1500,0.217200


TrainOutput(global_step=1563, training_loss=0.25673876522598704, metrics={'train_runtime': 1180.8825, 'train_samples_per_second': 21.171, 'train_steps_per_second': 1.324, 'total_flos': 3311684966400000.0, 'train_loss': 0.25673876522598704, 'epoch': 1.0})

**`TODO:`** Use `evaluate_model` to measure the performance of the post-trained model.

In [20]:
evaluate_model(model,tokenizer,test_loader)

100%|██████████| 1563/1563 [06:20<00:00,  4.11it/s]

Accuracy:  92.25999999999999 %
 -- Positive Accuracy:  91.08000000000001 %
 -- Negative Accuracy:  93.44 %
F1 score:  0.9216757741347905


(tensor([0, 0, 0,  ..., 1, 1, 1]),
 tensor([0, 0, 0,  ..., 1, 1, 1]),
 0.9226,
 0.9216757741347905)

# Task-B: Machine Translation

### `text_target` in Tokenizers

- The text_target argument is used when working with sequence-to-sequence (encoder–decoder) models such as T5, BART, mBART, or mT5.
- It allows you to tokenize the target/output text (e.g. a translation or summary) alongside the input text in a single call to the tokenizer.
- The tokenized targets are stored under the key labels, which the model uses during training for loss computation.

### AutoModelForSeq2SeqLM
- `AutoModelForSeq2SeqLM` is a Hugging Face class that loads a model configured for sequence-to-sequence tasks (e.g., machine translation, text summarization, question answering, text generation with input-output pairs).
- These models typically take in a sequence as input (e.g., a sentence or paragraph) and generate a new sequence as output (e.g., a translated or summarized version).

Note: The specific tokenizer will require you to `pip install protobuf`.


**`TODO:`** Load the tokenizer and the pretrained model for sequence-to-sequence tasks for `google/mt5-small`.

In [21]:
tokenizer = AutoTokenizer.from_pretrained('google/mt5-small')
model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small')

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

**`TODO:`** Use the tokenizer to tokenize both an input and a target in one go. You can use "Hello" for input and "Bonjour" for output.

In [23]:
input_text = "Hello"
target_text = "Bonjour"
model_inputs = tokenizer(
    input_text,
    text_target=target_text,
    truncation=True,
    padding=True,
    return_tensors="pt"
)
print(model_inputs)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'input_ids': tensor([[30273,     1]]), 'attention_mask': tensor([[1, 1]]), 'labels': tensor([[  259, 66392,     1]])}


**`TODO:`** Unzip the `wmt14_fr_en_10k` dataset and load it into a HF `Dataset`. Split the data into a training and a validation set. To minimize the procedure, use the `.select(N)` method of the `Dataset` to use only 3000 samples for training and 50 for validation. Then, print the data, to see their feaures and ensure the number of samples.

In [28]:
!unzip wmt14_fr_en_10k.zip


Archive:  wmt14_fr_en_10k.zip
   creating: wmt14_fr_en_10k/
  inflating: __MACOSX/._wmt14_fr_en_10k  
  inflating: wmt14_fr_en_10k/.DS_Store  
  inflating: __MACOSX/wmt14_fr_en_10k/._.DS_Store  
  inflating: wmt14_fr_en_10k/dataset_dict.json  
  inflating: __MACOSX/wmt14_fr_en_10k/._dataset_dict.json  
   creating: wmt14_fr_en_10k/test/
  inflating: __MACOSX/wmt14_fr_en_10k/._test  
   creating: wmt14_fr_en_10k/train/
  inflating: __MACOSX/wmt14_fr_en_10k/._train  
   creating: wmt14_fr_en_10k/validation/
  inflating: __MACOSX/wmt14_fr_en_10k/._validation  
  inflating: wmt14_fr_en_10k/test/state.json  
  inflating: __MACOSX/wmt14_fr_en_10k/test/._state.json  
  inflating: wmt14_fr_en_10k/test/dataset_info.json  
  inflating: __MACOSX/wmt14_fr_en_10k/test/._dataset_info.json  
  inflating: wmt14_fr_en_10k/test/data-00000-of-00001.arrow  
  inflating: __MACOSX/wmt14_fr_en_10k/test/._data-00000-of-00001.arrow  
  inflating: wmt14_fr_en_10k/train/state.json  
  inflating: __MACOSX/wmt14_f

**`TODO:`** Define a function that receives some samples and then uses the tokenizer we have defined to tokenize the samples. Append the phrase `"translate English to French: "` to the inputs. Tokenize the targets (french translation) as well. Use the `Dataset.map`method that we have previously discussed to apply your function to all of the data.

In [29]:
dataset = load_from_disk('wmt14_fr_en_10k')

In [32]:
dataset['train'][0]

{'translation': {'en': 'As Mauritius becomes a knowledge-intensive economy, with the development of the information and communications technologies (ICT) sector and the vision of turning Mauritius into a cyber-island, there is a real risk of exacerbating the current skills mismatch - especially among women - unless educational reforms are implemented to cater to the new exigencies of the labour market.',
  'fr': "À mesure que Maurice devient une économie à forte intensité de savoir, que le secteur des technologies de l'information et de la communication (TIC) se développe avec l'idée de faire de Maurice une cyber-île, l'écart actuel des connaissances risque fort de se creuser, surtout chez les femmes, sauf si les réformes de l'enseignement sont adaptées aux impératifs neufs du marché du travail."}}

In [42]:
train_data = dataset['train'].select(range(3000))
test_data = dataset['test'].select(range(50))

In [70]:
def tokenize_function_2(example):
    # Prepend prompt to input
    source = "translate English to French: " + example["translation"]["en"]
    target = example["translation"]["fr"]

    # Tokenize both input and target
    model_inputs = tokenizer(
        source,
        text_target=target,
        truncation=True,
        max_length=128
    )

    return model_inputs



In [71]:
train_dataset = train_data.map(tokenize_function_2, batched=False)
eval_dataset = test_data.map(tokenize_function_2, batched=False)


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

### Evaluate
- Hugging Face's library for evaluations.
- By importing `evaluate` the library provides ready-to-use implementations of common metrics (accuracy, F1, BLEU, ROUGE, etc.).
- The `evaluate.load("sacrebleu")` method, loads the SacreBLEU metric, a standard metric for evaluating machine translation quality. Give it a try in the following example. Any ideas about what the results could mean?

More information regarding HF Evaluate can be found in the following [documentation](https://huggingface.co/docs/evaluate/en/index).

In [37]:
sacrebleu = evaluate.load("sacrebleu")

predictions = ["the cat is on the mat"]
references = [["there is a cat on the mat"]]

results = sacrebleu.compute(predictions=predictions, references=references)
print(results)

{'score': 29.05925408079185, 'counts': [5, 2, 1, 0], 'totals': [6, 5, 4, 3], 'precisions': [83.33333333333333, 40.0, 25.0, 16.666666666666668], 'bp': 0.846481724890614, 'sys_len': 6, 'ref_len': 7}


In [36]:
%pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.4 MB/s eta 0:00:00


### Data Collators (continued)
- The DataCollatorForSeq2Seq is a special collator for sequence-to-sequence tasks (like translation or summarization). It not only handles dynamic padding, but also makes sure that both the inputs and the labels (decoder side) are correctly padded. It can also prepare the labels for the model’s loss function (e.g. replacing padding tokens with -100 so they’re ignored during loss computation).
- Because of this, the model as well as the tokenizer are required to initialize it: it uses the model’s configuration (e.g. label_pad_token_id, eos_token_id) to ensure that the labels it produces match exactly what the model expects for training and loss calculation.

In [68]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, max_length=128)

Same as before, we need to be able to evaluate our model. Below is the evaluation method that we have chosen.

In [39]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Convert token IDs back to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels as the padding token ID, then decode
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # SacreBLEU expects a list of prediction strings, plus a list of lists for references
    results = sacrebleu.compute(
        predictions=decoded_preds,
        references=[[lbl] for lbl in decoded_labels]
    )
    return {"bleu": results["score"]}


**`TODO:`** In the same way as previously, define the `Seq2SeqTrainingArguments`, the `Seq2SeqTrainer` and train the model. Once it's done evaluate the final model.

Notes:
- You now have a validation split as well which the trainer can use.
- The evaluation method given to you `compute_metrics` can also be given to the `Seq2SeqTrainer` for the same reason.

In [46]:
output_dir_name = "wmt14-finetuned-mt5"

training_args_seq2seq = Seq2SeqTrainingArguments(
    output_dir=output_dir_name,          # Where to save model checkpoints and logs
    learning_rate=2e-5,                  # Step size for the optimizer (how fast the model learns)
    per_device_train_batch_size=16,      # Training batch size per GPU/CPU device
    per_device_eval_batch_size=16,       # Evaluation batch size per GPU/CPU device
    num_train_epochs=1,                  # Number of times to iterate over the full training dataset
    weight_decay=0.01,                   # Strength of L2 regularization (helps prevent overfitting)
    save_strategy="epoch",               # When to save checkpoints ("epoch" = at the end of each epoch)
    push_to_hub=False,                   # Whether to push the model to the Hugging Face Hub
    report_to="none",                     # Where to report logs (e.g., "wandb", "tensorboard", "none")
    predict_with_generate=True # Enable generation for prediction
)

In [72]:


trainer_seq2seq = Seq2SeqTrainer(model = model,
                                 args = training_args_seq2seq,
                                 train_dataset = train_dataset,
                                 eval_dataset = eval_dataset,
                                 tokenizer = tokenizer,
                                 data_collator = data_collator,
                                 compute_metrics = compute_metrics)

/tmp/ipython-input-449176979.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_seq2seq = Seq2SeqTrainer(model = model,


In [73]:
trainer_seq2seq.train()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss


TrainOutput(global_step=188, training_loss=18.161714594414892, metrics={'train_runtime': 210.7972, 'train_samples_per_second': 14.232, 'train_steps_per_second': 0.892, 'total_flos': 302568748892160.0, 'train_loss': 18.161714594414892, 'epoch': 1.0})

In [77]:
trainer_seq2seq.evaluate()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


{'eval_loss': 10.544615745544434,
 'eval_bleu': 0.03797549195408434,
 'eval_runtime': 1.2964,
 'eval_samples_per_second': 38.569,
 'eval_steps_per_second': 3.086,
 'epoch': 1.0}